# FMA Small: cleaning, scaling, and reduction comparisons

This notebook implements further cleaning based on the results from the initial exploration.

**Decisions:** consolidate duplicate records; retain conflicting-title cases; do not impute clean values or remove unusual songs. Standard scaling is the baseline. Address skewed components/features; correlation filtering and PCA are both implemented as feature reduction alternatives.

In [ ]:
from pathlib import Path
import json
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/preprocessing.py").is_file()
             and (p / "data/fma_metadata/tracks.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the repository root or notebooks/ directory")
sys.path.insert(0, str(ROOT / "src"))
from preprocessing import run_pipeline, restore_representations, apply_power_transforms

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 25)
plt.rcParams.update({"figure.figsize": (12, 4), "figure.dpi": 110})
PROCESSED = ROOT / "data/processed"
REPORTS = ROOT / "reports/cleaning"
manifest = run_pipeline(ROOT)
display(pd.Series({k: v for k, v in manifest.items()
                   if k not in ["config", "versions", "constant_features_removed"]}).to_frame("value"))

## Saved datasets and track alignment

`features_clean.parquet` keeps untransformed values and all nonconstant audio columns. Metadata is stored separately. `metadata_all.parquet` and `track_mapping.parquet` retain all 8,000 source IDs, including consolidated records, so their titles and genres remain recoverable.

The source has no NaNs/infinities or constant columns. The pipeline checks this again and stops for review if new non-finite values appear. No imputation is needed and no rows are removed as outliers.

In [ ]:
clean = pd.read_parquet(PROCESSED / "features_clean.parquet")
metadata = pd.read_parquet(PROCESSED / "metadata_clean.parquet")
mapping = pd.read_parquet(PROCESSED / "track_mapping.parquet")
bundle = joblib.load(PROCESSED / "transformations.joblib")
train_ids = bundle["fit_track_ids"]
assert clean.index.equals(metadata.index)
assert mapping.canonical_track_id.isin(clean.index).all()
display(clean.iloc[:5, :8].T)
display(metadata["set__split"].value_counts().to_frame("tracks"))
display(mapping.loc[mapping.index != mapping.canonical_track_id])

## Conservative duplicate decisions

Exact numerical equality is necessary but not sufficient for consolidation. Require a matching normalized artist and title as well. Normalization handles Unicode, case, whitespace, and an explicit `Artist - ` title prefix. It does not infer that different titles are the same recording. A retained representative is chosen by split order (training, validation, test), then lowest track ID. This is a metadata-supported decision, not audio verification.

In [ ]:
duplicates = pd.read_csv(REPORTS / "duplicate_decisions.csv")
display(duplicates)
print("Remaining duplicate-vector groups crossing official splits:",
      manifest["duplicate_vector_groups_crossing_splits"])
display(duplicates.loc[duplicates.decision.eq("retain_unresolved")])

## Scaling versus correcting skew

StandardScaler subtracts the training mean and divides by the training standard deviation, giving comparable scales. **It does not remove skew.**

For the alternative representation, consider columns with training absolute skew above 2, more than 10 distinct values, and modal fraction below 99%. Fit a Yeo–Johnson transform per eligible column; it accommodates zero and negative values. Keep it only when training absolute skew improves, and reject numerical failures. Then standardize all columns. No clipping or track deletion occurs.

Nearly invariant columns are retained without power transformation and remain candidates for later domain review. Their rare deviations can still have large standardized values. Lower skew does not prove Gaussianity, better similarity rankings, or a better model. Residual skew is reported rather than hidden.

In [ ]:
transforms = pd.read_csv(REPORTS / "transform_decisions.csv")
comparison = pd.read_csv(REPORTS / "representation_comparison.csv")
display(comparison)
display(transforms.groupby(["decision", "reason"], dropna=False).size().to_frame("features"))
improved = transforms.loc[transforms.decision.eq("yeo_johnson")].copy()
improved["absolute_skew_reduction"] = (improved.training_skew_before.abs()
                                       - improved.training_skew_after.abs())
display(improved.sort_values("absolute_skew_reduction", ascending=False).head(12))
transformed = apply_power_transforms(clean, bundle["power_transformers"])
selected = improved.sort_values("absolute_skew_reduction", ascending=False).feature_name.head(3)
fig, axes = plt.subplots(len(selected), 2, figsize=(13, 3 * len(selected)), squeeze=False)
for row, name in enumerate(selected):
    for col, (values, label) in enumerate([(clean.loc[train_ids, name], "Raw"),
                                           (transformed.loc[train_ids, name], "Yeo–Johnson")]):
        axes[row, col].hist(values, bins=60)
        axes[row, col].set_title(f"{label}: {name}\nskew = {values.skew():.2f}", fontsize=10)
        axes[row, col].set_ylabel("Training tracks")
fig.tight_layout()
plt.show()

## Pearson filtering: keep an interpretable representative

Recompute Pearson correlations **on the transformed/scaled training matrix**, so decisions match the representation used later. The original exploration used raw values across the full Small subset, so counts need not match. Standard scaling itself does not change Pearson correlation; nonlinear power transforms can.

Drop a column only when it has absolute Pearson correlation at least 0.95 with an already retained column.

This heuristic favors interpretable summaries; it does not establish that discarded features have no unique information.

In [ ]:
drops = pd.read_csv(REPORTS / "correlation_drops.csv")
baseline_drops = pd.read_csv(REPORTS / "baseline_correlation_drops.csv")
filtered = pd.read_parquet(PROCESSED / "features_correlation_filtered.parquet")
print(f"Transformed: {clean.shape[1]} → {filtered.shape[1]} columns; {len(drops)} removed")
print(f"Standard-scaled baseline: {manifest['baseline_correlation_features']} columns retained")
display(drops.head(20))
corr = filtered.loc[train_ids].corr().abs().to_numpy()
assert not np.triu(corr >= .95, k=1).any()
print("Verified: no retained training pair has |r| >= 0.95")

## PCA: independent reduction alternative

Fit PCA to **all transformed/scaled features**, separately from correlation filtering, retaining at least 95% of training variance.`pca_loadings.csv` records all weights. A second PCA on the standard-scaled baseline compares dimensionality without skew transformations.

The 95% target concerns variance in the chosen scaled feature space, not 95% of musical information. Reconstruction metrics report how the same projection behaves on validation/test data; the retained-energy measure uses the training mean. It is not a recommendation-quality score or proof that PCA is the best representation.

In [ ]:
variance = pd.read_csv(REPORTS / "pca_variance.csv")
reconstruction = pd.read_csv(REPORTS / "pca_reconstruction.csv")
print("PCA components, transformed:", manifest["pca_components"])
print("PCA components, standard-scaled baseline:", manifest["baseline_pca_components"])
display(reconstruction)
fig, ax = plt.subplots()
ax.plot(variance.component, variance.cumulative_variance, label="Transformed/scaled PCA")
ax.plot(np.arange(1, bundle["baseline_pca"].n_components_ + 1),
        np.cumsum(bundle["baseline_pca"].explained_variance_ratio_), label="Standard-scaled PCA")
ax.axhline(.95, color="black", linestyle="--", label="95% training variance")
ax.set_xlabel("Number of components")
ax.set_ylabel("Cumulative explained variance")
ax.set_title("PCA dimensionality comparison")
ax.legend()
fig.tight_layout()
plt.show()
loadings = pd.read_csv(REPORTS / "pca_loadings.csv", index_col=0)
leading = []
for component, weights in loadings.head(3).iterrows():
    for name in weights.abs().nlargest(5).index:
        leading.append({"component": component, "feature_name": name, "loading": weights[name]})
display(pd.DataFrame(leading))

## Verify saved artifacts can be reused

Reload the saved bundle and apply it to the cleaned matrix, including an intentionally reversed input column order. The bundle restores feature order, applies training-fitted transforms, selects retained features, and projects onto PCA components. Missing/extra columns or non-finite inputs fail explicitly. This does not repeat duplicate detection; the track mapping documents that earlier step.

In [ ]:
restored = restore_representations(clean.loc[:, clean.columns[::-1]], bundle)
for name, expected in restored.items():
    saved = pd.read_parquet(PROCESSED / f"{name}.parquet")
    assert saved.index.equals(expected.index)
    assert saved.columns.equals(expected.columns)
    np.testing.assert_allclose(saved.to_numpy(), expected.to_numpy(), rtol=1e-10, atol=1e-10)
print("All saved feature representations reproduce after reloading fitted transformations.")
display(pd.DataFrame([
    {"file": p.name, "size_MiB": round(p.stat().st_size / 1024**2, 2)}
    for p in sorted(PROCESSED.iterdir()) if p.is_file()
]))

## Cleaning report and next analysis choice

The correlation-filtered representation is a useful starting point when named audio features matter. PCA is a compact comparison. Keep the standard-scaled baseline as a control, and assess these choices against the eventual musical-similarity objective before choosing one permanently. Neither approach is automatically better.

In [ ]:
report = f"""# FMA Small cleaning and representation report

## Cleaning decisions

- Input: {manifest['input_tracks']:,} tracks and {manifest['clean_features']} numeric feature columns.
- Removed {manifest['supported_duplicates_removed']} records with exactly equal feature vectors AND matching normalized artist/title. Artist-prefixed title variants are normalized. This is metadata-supported consolidation, not audio identity verification.
- Retained {manifest['clean_tracks']:,} tracks; {manifest['unresolved_tracks_retained']} conflicting-title tracks remain flagged across the Moon Veil and Janneh groups. There are {manifest['remaining_duplicate_vectors_beyond_first']} repeated vectors beyond their first occurrence among these retained rows.
- All original metadata and the source-to-canonical mapping are saved separately, preserving all 8,000 source IDs. Raw files remain unchanged.
- No imputation, constant-column removal, clipping, or outlier-row removal was necessary. Numerical features remain separate from metadata. The cleaned matrix retains raw units.
- Remaining exact-vector groups crossing official splits: {manifest['duplicate_vector_groups_crossing_splits']}.

## Scaling and skew

- All scalers, power transforms, correlation decisions, and PCA fit only the {manifest['fit_tracks']:,} retained training rows; validation/test receive those fitted transformations.
- StandardScaler provides a full-feature baseline; it changes scale but not skew.
- Selective Yeo–Johnson transforms were adopted for {manifest['power_transformed_features']} columns. Eligible columns have training |skew| > 2, >10 unique values, and modal fraction <99%; accept only improved training absolute skew and numerically finite outputs.
- Training columns with |skew| > 2 decreased from {manifest['training_skew_gt2_before']} to {manifest['training_skew_gt2_after']}. Residual skew and saturated columns remain visible in transform_decisions.csv. Reduced skew does not prove better musical similarity.

## Reduction comparisons

- Correlation filtering retained {manifest['correlation_features']} of {manifest['clean_features']} transformed/scaled columns at |Pearson r| >=0.95. Standard-scaled baseline filtering retained {manifest['baseline_correlation_features']} columns, potentially with different membership.
- Prefer mean, median, std, min, max, skew, then kurtosis; each dropped column maps to a surviving correlated representative. No retained training pair exceeds the threshold. The input feature order is preserved.
- Independent PCA on all transformed/scaled columns retained {manifest['pca_components']} components for {manifest['pca_training_variance']:.2%} of training variance; baseline standard-scaled PCA required {manifest['baseline_pca_components']} components for at least 95% variance.
- PCA is not chained after correlation filtering. Loadings and split-specific reconstruction metrics are saved. Variance is not a measure of preserved musical meaning.

```text
{reconstruction.to_string(index=False)}
```

## Artifacts and reproduction

Run all cells in notebooks/02_preprocessing.ipynb or run `.venv/bin/python src/preprocessing.py` from the repository root. Generated data in data/processed/ is ignored by Git; reports/cleaning/ holds the small audit artifacts. Re-running replaces generated outputs only.

- features_clean.parquet: cleaned rows, raw units, all numeric columns.
- features_scaled.parquet: StandardScaler baseline.
- features_transformed_scaled.parquet: selective Yeo–Johnson followed by StandardScaler.
- features_correlation_filtered.parquet: interpretable subset of the transformed/scaled matrix.
- features_pca.parquet: PCA alternative from the full transformed/scaled matrix.
- metadata_all.parquet, metadata_clean.parquet, track_mapping.parquet: metadata and original-to-canonical IDs.
- transformations.joblib: fitted transformations, selected columns, training IDs, PCA alternatives, and configuration. manifest.json records package versions and counts.

## Recommended use and remaining limits

Start with the {manifest['correlation_features']}-column correlation-filtered representation for interpretable analysis and compare the PCA representation when compactness matters. Keep the full baseline as a control. Investigate the eight conflicting-title tracks using targeted audio checks before removing them; this workflow never opens audio. Review near-invariant columns and residual skew before treating standardized distances as musically meaningful. These are preprocessing alternatives, not evaluated recommendation models.
"""
(REPORTS / "cleaning_report.md").write_text(report)
# display(Markdown(report))